In [8]:
# 1 Mask-1: Prep AV and VidaAtree and return offaw_npa and avseg_npf

import SimpleITK as sitk
import numpy as np

# Have done: Case-38-KPSVJK, Case-71-ESFJJW, Case-79-EERQOV --- Case-56-FAPNCX, Case-50-UXGSDR
CaseNumStr = '79'

offaw_im     = sitk.ReadImage(r'Z:\E-Masks-Analyses-&Vida\VIDA-Analyses-IndividualAndBatch\A-Jensen-Vida-Analysis\A-Full-Analysis\Case-79-EERQOV\Analysis\ZUNU_vida-airtree.hdr' )
offaw_npa    = sitk.GetArrayFromImage(offaw_im)

offawclr_im  = sitk.ReadImage(r'Z:\E-Masks-Analyses-&Vida\VIDA-Analyses-IndividualAndBatch\A-Jensen-Vida-Analysis\A-Full-Analysis\Case-79-EERQOV\Analysis\ZUNU_vida-aircolor.hdr')
offawclr_npa = sitk.GetArrayFromImage(offawclr_im)

avseg_im  = sitk.ReadImage(r'Z:\E-Masks-Analyses-&Vida\VIDA-Analyses-IndividualAndBatch\A-Jensen-Vida-Analysis\A-Full-Analysis\Case-79-EERQOV\Analysis\ExtraFiles\S-Final-2-label-At-Vx32.nrrd')
avseg_npa = sitk.GetArrayFromImage(avseg_im)
avseg_npf = np.flip(avseg_npa, axis=0)
avseg_im_mv = sitk.GetImageFromArray(avseg_npf)
avseg_im_mv.CopyInformation(offaw_im)
PthToAVLabelAtOffVidaAw = r'Z:\E-Masks-Analyses-&Vida\VIDA-Analyses-IndividualAndBatch\A-Jensen-Vida-Analysis\A-Full-Analysis\Case-79-EERQOV\Analysis\ExtraFiles\SFinal2Label-Brough2-OffVidaAw.nrrd'
sitk.WriteImage(avseg_im_mv, PthToAVLabelAtOffVidaAw)
xml_file = r'Z:\E-Masks-Analyses-&Vida\VIDA-Analyses-IndividualAndBatch\A-Jensen-Vida-Analysis\A-Full-Analysis\Case-79-EERQOV\Analysis\ZUNU_vida-xmlTree.xml'

offaw_npa    = np.transpose(offaw_npa,    (2, 1, 0))  # Now in (x, y, z) order
offawclr_npa = np.transpose(offawclr_npa, (2, 1, 0))  # Now in (x, y, z) order
avseg_npf    = np.transpose(avseg_npf,    (2, 1, 0))  # Now in (x, y, z) order

In [9]:
# 2 Mask-2: Create np offaw-aseg masks

offaw_npa    = (offaw_npa  > 0).astype(np.uint8 )
offawclr_npa = (offawclr_npa  ).astype(np.uint16)
aseg_npf     = (avseg_npf == 1).astype(np.uint8 )

In [10]:
# 3 Graph-1: Create the AwDigraph from VIDA xml

import xml.etree.ElementTree as ET
import networkx as nx

def create_digraph_from_xml(xml_file):
    
    # Parse the XML file
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Create a directed graph
    AwDigraph = nx.DiGraph()

    # Process Branchpoints
    node_id_to_position = {} # To keep track of id to position mapping
  
    for bp in root.findall('.//Branchpoints/BP'):
        node_id  =  int(bp.get('id'))
        position = (int(bp.get('x')), int(bp.get('y')), int(bp.get('z')))
        node_id_to_position[node_id] = position
        AwDigraph.add_node(position)  # Use position as node identifier

    # Process Links
    for link in root.findall('.//Links/Link'):
        link_id = int(link.get('id' ))
        bp1_id  = int(link.get('bp1'))
        bp2_id  = int(link.get('bp2'))
        mean_luminal_diameter = link.get('meanLuminalDiameter')

        # Convert positions
        bp1_position = node_id_to_position[bp1_id]
        bp2_position = node_id_to_position[bp2_id]

        if mean_luminal_diameter == 'NA':
            mean_luminal_diameter = None
        else:
            mean_luminal_diameter = float(mean_luminal_diameter)

        AwDigraph.add_edge(bp1_position, bp2_position, meanLuminalDiameter=mean_luminal_diameter,
                           LinkId=link_id)
        
    # Process Root
    for rt in root.findall('.//RootNodes/RootNode'):
        root_bp_id = int(rt.get('branchpId'))
        AwRoot = node_id_to_position[root_bp_id]

    # Process SegmentNames
    AnatName2LinkId = {} # To keep track of anatomicalName to linkIds mapping

    for sn in root.findall('.//SegmentNames/SegmentName'):
        sn_anatomicalName = sn.get('anatomicalName')
        sn_linkIds = int(sn.get('linkIds'))
        AnatName2LinkId[sn_anatomicalName] = sn_linkIds

    return AwDigraph, AwRoot, AnatName2LinkId

AwDigraph, AwRoot, AnatName2LinkId = create_digraph_from_xml(xml_file)

In [24]:
# 2 Loads the AwATrainSet
import pickle

def LoadTrainingSet(PTH):

    with open(PTH, 'rb') as f: AwATrainSet = pickle.load(f)    

    return AwATrainSet

TrainSetPath = r'Z:\F-People\Nate\FileExchange-DeleteWhenDonePls\AwATrainSetSaves\Temp-AwATrainSet-Save-32.pkl'
AwATrainSet = LoadTrainingSet(TrainSetPath)

In [25]:
AwATrainSet

[['Case-38', 'Trachea', 1, '', ''],
 ['Case-38', 'RMB', 99, '', ''],
 ['Case-38', 'LMB', 2, '', ''],
 ['Case-38', 'BronInt', 100, '', ''],
 ['Case-38', 'RUL', 161, '', ''],
 ['Case-38', 'RB4+5', 154, '', ''],
 ['Case-38', 'RLL7', 104, '', ''],
 ['Case-38', 'LUL', 84, 'N', 'NoExactCounterpart'],
 ['Case-38', 'LLB6', 10, 'M', 'BetterWouldBe4+10'],
 ['Case-38', 'RB1', 170, 'E', ''],
 ['Case-38',
  'RB4',
  105,
  'M',
  'ButAlsoPartlyWith154-CorrespondingToTwoParallelBranches'],
 ['Case-38', 'RB10', 135, 'EM', 'NotParallel-But1To1Correspondence'],
 ['Case-38', 'LB10', 35, 'MN', 'SortOf!'],
 ['Case-38', 'rb1_a', 172, 'E', ''],
 ['Case-38', 'rb1_b', 171, 'E', ''],
 ['Case-38', 'rb4_a', 107, 'EM', 'AlmostExactCorr'],
 ['Case-38', 'rb4_b', 106, 'EM', 'AlmostExactCorr'],
 ['Case-38', 'rb10_a', 136, 'EM', ''],
 ['Case-38', 'rb10_b', 137, 'EM', ''],
 ['Case-38', 'lb1_a', 73, 'N', 'AndMaybeALittleOf6'],
 ['Case-38', 'lb1_b', 85, 'EM', ''],
 ['Case-38', 'lb10_a', 36, 'EM', ''],
 ['Case-38', 'lb10_

In [13]:
# 4 Graph-2: Import AGraph/ADiGraph/ARoot/atreepartitioned

AVarDumpOrLoad = 'LoadFromFile' # 'Dump2File' # -----------------------------------------------------------------

Prefix = r'Z:\F-People\Nate\FileExchange-DeleteWhenDonePls\MovedGraphArtifacts'

if AVarDumpOrLoad == 'Dump2File':
    
    with open(Prefix + f'Temp-Case{CaseNumStr}Moved-A-Graph.pkl'         , 'wb') as f: pickle.dump(graph,             f)
    with open(Prefix + f'Temp-Case{CaseNumStr}Moved-A-DiGraph.pkl'       , 'wb') as f: pickle.dump(digraph,           f)
    with open(Prefix + f'Temp-Case{CaseNumStr}Moved-A-Root.pkl'          , 'wb') as f: pickle.dump(root,              f)
    with open(Prefix + f'Temp-Case{CaseNumStr}Moved-ATreePartitioned.pkl', 'wb') as f: pickle.dump(atreepartitioned,  f)
    AGraph   = graph
    ADiGraph = digraph
    ARoot    = root
    del graph, digraph, root

elif AVarDumpOrLoad == 'LoadFromFile':

    with open(Prefix + f'\Temp-Case{CaseNumStr}Moved-A-Graph.pkl'         , 'rb') as f: AGraph           = pickle.load(f)
    with open(Prefix + f'\Temp-Case{CaseNumStr}Moved-A-DiGraph.pkl'       , 'rb') as f: ADiGraph         = pickle.load(f)
    with open(Prefix + f'\Temp-Case{CaseNumStr}Moved-A-Root.pkl'          , 'rb') as f: ARoot            = pickle.load(f)
    with open(Prefix + f'\Temp-Case{CaseNumStr}Moved-ATreePartitioned.pkl', 'rb') as f: atreepartitioned = pickle.load(f)

In [22]:
# To construct Aw-A training set: 1-2-3:
# 1- Pick Aw label + Go through SegList and populate AwTargetEdges and ATargetEdges

LinkId2Edge = lambda Graph,LinkId : [(nd1, nd2) for nd1, nd2, data in Graph.edges(data=True) if data['LinkId'] == LinkId]

# PickStrategy = 1 # argmin(dist/abs(orienmatch))
ATargetEdges  = []
AwTargetEdges = []
offawclr_npa_RedSegValues = []
atreeparti_RedSegValues = []

SegList   = ['Trachea']
# SegList = ['Trachea', 'RMB', 'LMB', 'BronInt', 'RUL', 'RB4+5', 'RLL7', 'LUL', 'LLB6', 'RB1', 'RB4', 'RB10', 'LB10',
#            'rb1_a', 'rb1_b', 'rb4_a', 'rb4_b', 'rb10_a', 'rb10_b', 'lb1_a', 'lb1_b', 'lb10_a', 'lb10_b', 'RB6']

for Seg in SegList:

    AW_SEG = AnatName2LinkId[Seg]
    # offawclr_npa_RedSegValues.append(AW_SEG)
    AwTargetEdge = LinkId2Edge(AwDigraph, AW_SEG)
    AwTargetEdges.append(AwTargetEdge[0])    

    # AFeaturesMatrixForAwEdge = ABranchFeaturesForAwTargetEdgeSay6x186(AwTargetEdge, AwDigraph, ADiGraph)
    # ATargetEdgeIndxN = PickTargetAEdgeIndxUsingAllStrategies(Seg_Index, AFeaturesMatrixForAwEdge)                       

    ATargetEdge = list(ADiGraph.edges)[ATargetEdgeIndxN[PickStrategy]]
    # ATargetEdge = list(ADiGraph.edges)[0]
    # atreeparti_RedSegValues.append(ADiGraph[ATargetEdge[0]][ATargetEdge[1]]['AssignedEdgeNumber'])
    ATargetEdges.append(ATargetEdge)

# ATargetEdges = [1] ########################################################## Added here to suppress red 'A' edge links

In [23]:
# 2- Orange/Red Mayavi

from mayavi import mlab
import numpy as np
import winsound

IsInEdgeList = lambda Edge, EdgeList: (Edge in EdgeList or Edge[::-1] in EdgeList)
fig = mlab.figure(size=(1400, 1400), bgcolor=(1, 1, 1)) # size=(3820, 2050)

DecimFac = 2 # DecimationFactor
SCL = 6.0 # nodes_scale_factor
OPC = 1   # 0.5 # 0.01 # 0.2 # contour3d red opacities
ELO = 0.9 # 0 # edge_line_opacities   ################ Change for AllOpaqueRed vis
GNO = 0.5 # 0 # graph_node_opacities  ################ Change for AllOpaqueRed vis
LNW = 9.0 # edge_line_width
RTO = 0   # 0.3 # root_opacities      ################ Change for AllOpaqueRed vis


## Add AwRedNodes
rednodes   = list(set([node for edge in AwTargetEdges for node in edge]))
dcrednodes = [tuple(i/DecimFac for i in node) for node in rednodes]
if dcrednodes:
    x, y, z = zip(*dcrednodes)
    mlab.points3d(x, y, z, scale_factor=SCL /DecimFac , color=(1, 0, 0), opacity=GNO)

# Add AwBlackNodes
allother   = [node for node in list(AwDigraph.nodes) if node not in set(rednodes)]
dcallother = [tuple(i/DecimFac for i in node) for node in allother]
x, y, z = zip(*dcallother)
mlab.points3d(x, y, z, scale_factor=SCL/DecimFac , color=(255/255, 192/255,   0/255), opacity=GNO)

# Add AwRedEdges
for edge in AwTargetEdges:
    mlab.plot3d([edge[0][0]/DecimFac, edge[1][0]/DecimFac],
                [edge[0][1]/DecimFac, edge[1][1]/DecimFac],
                [edge[0][2]/DecimFac, edge[1][2]/DecimFac],
                tube_radius=None, color=(1, 0, 0), opacity=ELO, line_width=LNW)

# Add AwBlackEdges
edge_coords_other = np.array([np.nan, np.nan, np.nan]) # To start stacking
for edge in AwDigraph.edges:
    if not IsInEdgeList(edge, AwTargetEdges):
        edge_coords_other = np.vstack((edge_coords_other, edge[0], edge[1], [np.nan, np.nan, np.nan]))
edge_coords_other = edge_coords_other[1:]/DecimFac # Remove the first row of NaNs
mlab.plot3d(edge_coords_other[:, 0], edge_coords_other[:, 1], edge_coords_other[:, 2],
            tube_radius=None, color=(255/255, 192/255,   0/255), opacity=ELO, line_width=LNW)

# Add AwRoot
if AwRoot: mlab.points3d(AwRoot[0]/DecimFac, AwRoot[1]/DecimFac, AwRoot[2]/DecimFac,
                        scale_factor=36/DecimFac, color=(255/255, 192/255,   0/255), opacity=RTO)

# Add AwContour3d
Vlm = offawclr_npa[::2,::2,::2]
offawclr_npa_RedSegValues = [] ######################## + Two commented lines below: added to suppress OpaqueRedSegs
RedMaskB   = np.isin(Vlm, offawclr_npa_RedSegValues) # True/False
# RedMask    = RedMaskB.astype(int) # 1/0
OrangeMask = np.where((Vlm != 0) & np.logical_not(RedMaskB), 1, 0) # 1/0
# mlab.contour3d(RedMask   , contours=[0.5], opacity=OPC, color=(1      , 0      , 0      ))
mlab.contour3d(OrangeMask, contours=[0.5], opacity=0.2, color=(255/255, 192/255,   0/255))


## Add ARedNodes
rednodes   = list(set([node for edge in ATargetEdges for node in edge]))
dcrednodes = [tuple(i/DecimFac for i in node) for node in rednodes]
if dcrednodes:
    x, y, z = zip(*dcrednodes)
    mlab.points3d(x, y, z, scale_factor=SCL /DecimFac , color=(1, 0, 0), opacity=GNO)

# Add ABlackNodes
allother   = [node for node in list(AGraph.nodes) if node not in set(rednodes)]
dcallother = [tuple(i/DecimFac for i in node) for node in allother]
x, y, z = zip(*dcallother)
mlab.points3d(x, y, z, scale_factor=SCL/DecimFac , color=(0      , 112/255, 192/255), opacity=GNO)

# Add ARedEdges
for edge in ATargetEdges:
    mlab.plot3d([edge[0][0]/DecimFac, edge[1][0]/DecimFac],
                [edge[0][1]/DecimFac, edge[1][1]/DecimFac],
                [edge[0][2]/DecimFac, edge[1][2]/DecimFac],
                tube_radius=None, color=(1, 0, 0), opacity=ELO, line_width=LNW)

# Add ABlackEdges
edge_coords_other = np.array([np.nan, np.nan, np.nan]) # To start stacking
for edge in AGraph.edges:
    if not IsInEdgeList(edge, ATargetEdges):
        edge_coords_other = np.vstack((edge_coords_other, edge[0], edge[1], [np.nan, np.nan, np.nan]))
edge_coords_other = edge_coords_other[1:]/DecimFac # Remove the first row of NaNs
mlab.plot3d(edge_coords_other[:, 0], edge_coords_other[:, 1], edge_coords_other[:, 2],
            tube_radius=None, color=(0      , 112/255, 192/255), opacity=ELO, line_width=LNW)

# Add ARoot
if ARoot: mlab.points3d(ARoot[0]/DecimFac, ARoot[1]/DecimFac, ARoot[2]/DecimFac,
                        scale_factor=36/DecimFac, color=(0      , 112/255, 192/255), opacity=RTO)

# Add AContour3d
Vlm = atreepartitioned[::2,::2,::2]
atreeparti_RedSegValues = [] ######################## + Two commented lines below: added to suppress OpaqueRedSegs
RedMaskB = np.isin(Vlm, atreeparti_RedSegValues) # True/False
# RedMask  = RedMaskB.astype(int) # 1/0
BlueMask = np.where((Vlm != 0) & np.logical_not(RedMaskB), 1, 0)
# mlab.contour3d(RedMask , contours=[0.5], opacity=OPC, color=(1      , 0      , 0      ))
mlab.contour3d(BlueMask, contours=[0.5], opacity=0.2, color=(0      , 112/255, 192/255))


# Display 'AssignedEdgeNumber' labels for 'A' edges
for edge in ADiGraph.edges:
    start_point = edge[0]
    end_point   = edge[1]

    # Calculate mid-point of the edge for placing the EdgeNum label
    mid_x = (start_point[0] + end_point[0]) / 2 / DecimFac
    mid_y = (start_point[1] + end_point[1]) / 2 / DecimFac
    mid_z = (start_point[2] + end_point[2]) / 2 / DecimFac
    edge_num = ADiGraph.edges[edge]['AssignedEdgeNumber']

    # Place EdgeNum text near the edge
    mlab.text3d(mid_x, mid_y, mid_z, str(edge_num), scale=(2, 2, 2), color=(0, 0, 0))


winsound.Beep(1000, 80)
mlab.show()